In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset=pd.read_csv('Social_Network_Ads.csv')

In [3]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [4]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [6]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [7]:
independent=dataset[['Age', 'EstimatedSalary','Gender_Male']]

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent=dataset['Purchased']

In [10]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size = 1/3, random_state = 0)

In [13]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [14]:
from sklearn.tree import DecisionTreeClassifier

In [16]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': [None, 'sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}


grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
grid.fit(X_train, y_train) 

Fitting 5 folds for each of 810 candidates, totalling 4050 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'class_weight': [None, 'balanced'], 'criterion': ['gini', 'entropy', ...], 'max_depth': [5, 10, ...], 'max_features': [None, 'sqrt', ...], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [17]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_

In [18]:
re

{'mean_fit_time': array([0.00367904, 0.00377049, 0.00269051, 0.00216331, 0.00255284,
        0.00212836, 0.0022964 , 0.00198398, 0.00208344, 0.00211458,
        0.00183859, 0.00190902, 0.0019732 , 0.00199413, 0.00186357,
        0.0018487 , 0.00199099, 0.00175743, 0.00194368, 0.00185957,
        0.00190043, 0.00193338, 0.0017076 , 0.00146523, 0.00180283,
        0.00183029, 0.00203295, 0.00165706, 0.00215406, 0.00200925,
        0.0019557 , 0.00220199, 0.00189638, 0.00200825, 0.00191483,
        0.00192003, 0.00196152, 0.00151873, 0.00204639, 0.00187759,
        0.00181437, 0.00201206, 0.00175858, 0.00172615, 0.00197191,
        0.00170379, 0.00169482, 0.0018683 , 0.00186901, 0.00195117,
        0.00189323, 0.00144696, 0.00166712, 0.00212069, 0.0020298 ,
        0.00210319, 0.00209947, 0.00187912, 0.00189877, 0.00212412,
        0.00176654, 0.00160413, 0.00174184, 0.0016315 , 0.00230885,
        0.00192208, 0.00171914, 0.00198331, 0.00189142, 0.00222254,
        0.00208554, 0.00181227,

In [19]:
grid_predictions = grid.predict(X_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [20]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 10}: 0.9043535442887727


In [21]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[74 11]
 [ 2 47]]


In [22]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.97      0.87      0.92        85
           1       0.81      0.96      0.88        49

    accuracy                           0.90       134
   macro avg       0.89      0.91      0.90       134
weighted avg       0.91      0.90      0.90       134



In [23]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])#[:,1] : → all rows,1 → second column (index starts from 0),Give me all rows, but only column 1 (positive class probability)

0.9561824729891957

In [24]:
table=pd.DataFrame.from_dict(re)

In [25]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,param_max_features,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003679,0.000937,0.005343,0.000615,None,gini,5,None,1,2,"{'class_weight': None, 'criterion': 'gini', 'm...",0.787654,0.868752,0.850543,0.906166,0.923510,0.867325,0.047532,451
1,0.003770,0.000359,0.005664,0.000650,None,gini,5,None,1,5,"{'class_weight': None, 'criterion': 'gini', 'm...",0.787654,0.868752,0.850543,0.925272,0.923510,0.871146,0.051132,362
2,0.002691,0.000356,0.004944,0.000509,None,gini,5,None,1,10,"{'class_weight': None, 'criterion': 'gini', 'm...",0.787654,0.868752,0.870362,0.925272,0.943041,0.879016,0.054343,246
3,0.002163,0.000250,0.005481,0.000662,None,gini,5,None,2,2,"{'class_weight': None, 'criterion': 'gini', 'm...",0.787654,0.850809,0.832483,0.906166,0.903610,0.856144,0.044798,610
4,0.002553,0.000261,0.004477,0.000526,None,gini,5,None,2,5,"{'class_weight': None, 'criterion': 'gini', 'm...",0.787654,0.850809,0.850543,0.906166,0.903610,0.859757,0.043453,559
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,0.003089,0.000204,0.004656,0.000526,balanced,log_loss,None,log2,2,5,"{'class_weight': 'balanced', 'criterion': 'log...",0.774691,0.834012,0.833323,0.787943,0.943041,0.834602,0.059207,776
806,0.002952,0.000244,0.004713,0.000186,balanced,log_loss,None,log2,2,10,"{'class_weight': 'balanced', 'criterion': 'log...",0.814815,0.870047,0.815149,0.852044,0.962636,0.862938,0.054237,517
807,0.003009,0.000201,0.004671,0.000222,balanced,log_loss,None,log2,5,2,"{'class_weight': 'balanced', 'criterion': 'log...",0.814815,0.870047,0.777291,0.907402,0.981233,0.870158,0.071291,370
808,0.002926,0.000206,0.004672,0.000349,balanced,log_loss,None,log2,5,5,"{'class_weight': 'balanced', 'criterion': 'log...",0.851852,0.889022,0.852044,0.962636,0.981233,0.907358,0.054754,3
